In [ ]:
#Q2. 주어진 URL을 이용하여, HTML을 다운받아 분석하고, 학과의 이름과 URL을 파싱하는 코드를 작성하시오.
# URL : https://www.duksung.ac.kr/contents/contents.do?ciIdx=3592&menuId=1167

Q2. 주어진 URL을 이용하여, HTML을 다운받아 분석하고, 학과의 이름과 URL을 파싱하는 코드를 작성하시오.

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# 대상 URL
base_url = "https://www.duksung.ac.kr"
target_url = "https://www.duksung.ac.kr/contents/contents.do?ciIdx=3592&menuId=1167"

# HTTP GET 요청
response = requests.get(target_url)
response.raise_for_status()  # 요청이 성공했는지 확인

# BeautifulSoup 객체 생성
soup = BeautifulSoup(response.text, "html.parser")
print(soup.prettify())
# 학과 목록이 포함된 영역 찾기

# 학과 목록에 있는 'a' 태그 찾기

<!DOCTYPE html>
<!--[if lt IE 7 ]><html lang="ko" class="ie6"><![endif]-->
<!--[if IE 7 ]>   <html lang="ko" class="ie7"><![endif]-->
<!--[if IE 8 ]>   <html lang="ko" class="ie8"><![endif]-->
<!--[if IE 9 ]>   <html lang="ko" class="ie9"><![endif]-->
<!--[if (gt IE 9)|!(IE)]><!-->
<html class="" lang="ko">
 <!--<![endif]-->
 <head>
  <meta charset="utf-8"/>
  <meta content="IE=edge" http-equiv="X-UA-Compatible">
   <title>
    대학 | 교육 | 덕성여자대학교
   </title>
   <meta content="font-src 'self' https://fonts.gstatic.com" http-equiv="content-security-policy"/>
   <meta content="style-src 'self' 'unsafe-inline' https://t1.daumcdn.net" http-equiv="content-security-policy"/>
   <meta content="img-src 'self' https://dis.duksung.ac.kr https://t1.daumcdn.net https://mts.daumcdn.net" http-equiv="content-security-policy"/>
   <meta content="connect-src 'self' https://txtnews.scrapmaster.co.kr" http-equiv="content-security-policy"/>
   <meta content="child-src 'self' https://www.youtube.com/ https:/

In [ ]:
majors = soup.select('li a') #<li> 안에 있는 모든 <a> 태그를 찾기

dept_dict = {} #정보를 저장할 빈 딕셔너리 생성 { '학과이름': 'URL' }

for major in majors:
    href = major.get('href') #<a> 태그의 링크 주소 가져오기
    name = major.get_text(strip=True) #<a> 태그 사이의 글자(학과명)를 가져오기

    if href and "majorInfo.do" in href: #링크에 'majorInfo.do'가 포함되어야 함
        if name.endswith(("전공", "학과", "과")) and not name.endswith("대학"): #필터링 조건
            full_url = urljoin(base_url, href)

            dept_dict[name] = full_url #딕셔너리에 저장 (이름-키, URL-값)

# 딕셔너리 출력
for name, url in dept_dict.items():
    print(f"{name} -> {url}")

print("-" * 85)

print(f"총 전공 개수: {len(dept_dict)}개")

국어국문학전공 -> https://www.duksung.ac.kr/univ/majorInfo.do?miIdx=21&menuId=984
일어일문학전공 -> https://www.duksung.ac.kr/univ/majorInfo.do?miIdx=25&menuId=985
중어중문학전공 -> https://www.duksung.ac.kr/univ/majorInfo.do?miIdx=26&menuId=986
영어영문학전공 -> https://www.duksung.ac.kr/univ/majorInfo.do?miIdx=27&menuId=987
불어불문학전공 -> https://www.duksung.ac.kr/univ/majorInfo.do?miIdx=28&menuId=988
독어독문학전공 -> https://www.duksung.ac.kr/univ/majorInfo.do?miIdx=29&menuId=989
스페인어전공 -> https://www.duksung.ac.kr/univ/majorInfo.do?miIdx=30&menuId=990
사학전공 -> https://www.duksung.ac.kr/univ/majorInfo.do?miIdx=31&menuId=991
철학전공 -> https://www.duksung.ac.kr/univ/majorInfo.do?miIdx=32&menuId=992
미술사학전공 -> https://www.duksung.ac.kr/univ/majorInfo.do?miIdx=33&menuId=993
경영학전공 -> https://www.duksung.ac.kr/univ/majorInfo.do?miIdx=22&menuId=995
회계학전공 -> https://www.duksung.ac.kr/univ/majorInfo.do?miIdx=37&menuId=999
국제통상학전공 -> https://www.duksung.ac.kr/univ/majorInfo.do?miIdx=41&menuId=1003
법학전공 -> https://www.duksung.ac.kr/un

In [26]:
#학과별 교수님 성함 추출
import time

for name, url in dept_dict.items(): #'학과명:주소' 하나씩 꺼내기
    try:
        res = requests.get(url) #해당 학과 상세 페이지로 직접 접속
        res.encoding = res.apparent_encoding #한글 깨지지 않게 인코딩
        sub_soup = BeautifulSoup(res.text, "html.parser")

        # 교수 정보가 담긴 리스트 항목들
        prof_items = sub_soup.select('ul.ul_list.small li') #ul_list.small 클래스 안에 있는 모든 <li> 항목들을 리스트로 긁어오기(전화번호, 학위 다 포함)

        print(f"[{name}]") #학과 이름 출력

        found_any = False
        profs_seen = set()

        for item in prof_items:
            text = item.get_text(strip=True) #<li> 태그 안의 글자만 가져오기

            if ":" in text: #콜론이 있는 줄만 검사
                parts = text.split(":") # 콜론를 기준으로 문장 쪼개기
                title = parts[0].strip()  #콜론 왼쪽
                content = parts[1].strip() #콜론 오른쪽

                # 핵심 조건
                if "교수" in title: #왼쪽이 '교수'일 때만
                    if content and content not in profs_seen: #중복 출력 방지
                        print(f"- {content}") #교수님 성함 출력
                        profs_seen.add(content) #set에 add
                        found_any = True #결과가 없을 때

        if not found_any:
            print("-교수 정보를 찾을 수 없습니다.")

        print()
        time.sleep(0.3)

    except Exception as e:
          print(f"[{name}] 접속 에러 발생\n")

[국어국문학전공]
- 김은희
- 이은애
- 이명찬
- 양정호
- 김유진
- 김지은

[일어일문학전공]
- 오경
- 이광수
- 이상경
- 손재현
- 이이야마다케시
- 노주현
- 사까이마유미(Sakai Mayumi)
- 호규진
- 이정은

[중어중문학전공]
- 양오진
- 강춘화
- 김경남
- 위메이(Jiao Yumei)
- 박민준
- 오헌필

[영어영문학전공]
- 윤지관
- 김문규
- 정혜옥
- 김영미
- 윤희철
- 우정민
- 이수영

[불어불문학전공]
- 박혜영
- 차지영
- 앙토니 꼬레이야

[독어독문학전공]
- 조우호
- 곽정연
- Monika Moravkova

[스페인어전공]
- 전진재
- 권은희
- 이종득
- 라이먼(Raimon Blancafort Lopez)
- 김상윤

[사학전공]
- 한상권
- 이창신
- 최주희
- 김성운
- 김정신

[철학전공]
- 민형원
- 한우진
- 김제란

[미술사학전공]
- 최성은
- 박은순
- 정무정
- 이송란
- 정수희

[경영학전공]
- 김성철
- 김경묵
- 이상묵
- 노태협
- 장욱
- 유정민
- 정지용
- 유병희
- 나재석
- 이황희
- 김철규
- 강병국
- 조연진

[회계학전공]
- 안숙찬
- 송혁준
- 김이배
- 이문영
- 밭자르갈
- 김준철
- 유석원

[국제통상학전공]
- 박건영
- 이원정
- 원동환
- 김상만
- 백철우
- 조연성
- 남현정
- 박현용
- 강신호
- 정선기
- 조경규

[법학전공]
- 조인호
- 강수경
- 주한겸
- 김도훈
- 송일두

[사회학전공]
- 김종길
- 김은정
- 김두환

[문헌정보학전공]
- 박소연
- 정진수
- 정도헌
- 백재은

[심리학전공]
- 이종숙
- 김정호
- 오영희
- 주은선
- 김미리혜
- 김제중
- 최승원
- 김소연
- 박지영
- 강우선
- 김주영
- 오현주
- 한영경
- 우상우
- 박영현
- 장인희
- 정준용
- 조은영
- 오다영
- 임혜진
- 최설
- 강윤진

[아동가족학전공]
- 이옥
- 신화용
- 최미경
- 박우철
- 강수정
- 김지신
- 박민선
- 안수